Khushi Khatri BEB222 Experiment 6    
Aim:Study the different POS taggers and Perform POS tagging on the given text.

Introduction

Part-of-Speech (POS) tagging is the process of assigning a grammatical category — such as noun, verb, adjective, adverb, pronoun, preposition, or conjunction — to each word in a sentence based on both its definition and its context within that sentence. POS tagging is a fundamental step in Natural Language Processing (NLP), as it provides the syntactic foundation on which higher-level tasks such as parsing, named entity recognition, information extraction, and machine translation depend.

1. What is a POS Tag?

A POS tag labels a word with its grammatical role. For example, in the sentence "The cozy apartment was clean," the word "The" is tagged as a determiner, "cozy" and "clean" are tagged as adjectives, "apartment" is tagged as a singular noun, and "was" is tagged as a past-tense verb.

The same word can have different POS tags depending on context — for example, "book" is a noun in "I read a book" but a verb in "I will book a flight." This context-dependence is precisely why POS tagging requires more than a simple dictionary lookup; the surrounding words must be considered to resolve ambiguity.

In [2]:
import pandas as pd
from collections import Counter
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('tagsets')

from nltk.tokenize import word_tokenize, sent_tokenize

print('Setup complete.')

[nltk_data] Downloading package punkt to /home/computer/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package tagsets to /home/computer/nltk_data...


Setup complete.


[nltk_data]   Unzipping help/tagsets.zip.


In [3]:
df = pd.read_csv('Balanced_Airbnb_Reviews_Dataset.csv')
print('Shape:', df.shape)
df[['review_id', 'review_text']].head()

Shape: (15000, 42)


,review_id,review_text
0,369314882,Amazing stay! The place felt very cozy for 4 g...
1,490116563,It was okay for the price. Location in XIII Au...
2,582235668,Loved every minute of it. Our superhost was su...
3,68054683,Decent stay overall. Some things could be impr...
4,248483824,Reasonable for a short trip. Location in Long ...


In [4]:
def pos_tag_text(text):
    """Tokenize the text and return a list of (word, POS tag) pairs."""
    if not isinstance(text, str) or text.strip() == '':
        return []
    tokens = word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    return tagged

# Demo on a sample review
sample_text = df['review_text'].iloc[0]
print('Original Text:\n', sample_text, '\n')

tagged_tokens = pos_tag_text(sample_text)
print('POS Tagged Tokens:\n', tagged_tokens)

Original Text:
 Amazing stay! The place felt very cozy for 4 guests. Check-in was smooth and the amenities were exactly what we needed. 

POS Tagged Tokens:
 [('Amazing', 'VBG'), ('stay', 'NN'), ('!', '.'), ('The', 'DT'), ('place', 'NN'), ('felt', 'VBD'), ('very', 'RB'), ('cozy', 'JJ'), ('for', 'IN'), ('4', 'CD'), ('guests', 'NNS'), ('.', '.'), ('Check-in', 'NNP'), ('was', 'VBD'), ('smooth', 'JJ'), ('and', 'CC'), ('the', 'DT'), ('amenities', 'NNS'), ('were', 'VBD'), ('exactly', 'RB'), ('what', 'WP'), ('we', 'PRP'), ('needed', 'VBD'), ('.', '.')]


In [5]:
tagged_df = pd.DataFrame(tagged_tokens, columns=['word', 'pos_tag'])
tagged_df

,word,pos_tag
0,Amazing,VBG
1,stay,NN
2,!,.
3,The,DT
4,place,NN
5,felt,VBD
6,very,RB
7,cozy,JJ
8,for,IN
9,4,CD


In [6]:
def explain_tags(tagged_tokens):
    """Print a human-readable meaning for each POS tag used."""
    unique_tags = sorted(set(tag for _, tag in tagged_tokens))
    for tag in unique_tags:
        try:
            meaning = nltk.help.upenn_tagset(tag)
        except Exception:
            pass

explain_tags(tagged_tokens)

.: sentence terminator
    . ! ?
CC: conjunction, coordinating
    & 'n and both but either et for less minus neither nor or plus so
    therefore times v. versus vs. whether yet
CD: numeral, cardinal
    mid-1890 nine-thirty forty-two one-tenth ten million 0.5 one forty-
    seven 1987 twenty '79 zero two 78-degrees eighty-four IX '60s .025
    fifteen 271,124 dozen quintillion DM2,000 ...
DT: determiner
    all an another any both del each either every half la many much nary
    neither no some such that the them these this those
IN: preposition or conjunction, subordinating
    astride among uppon whether out inside pro despite on by throughout
    below within for towards near behind atop around if like until below
    next into if beside ...
JJ: adjective or numeral, ordinal
    third ill-mannered pre-war regrettable oiled calamitous first separable
    ectoplasmic battery-powered participatory fourth still-to-be-named
    multilingual multi-disciplinary ...
NN: noun, common, sing

In [7]:
def pos_tag_frequency(tagged_tokens):
    """Return a frequency count of each POS tag in the tagged text."""
    tags = [tag for _, tag in tagged_tokens]
    return Counter(tags)

tag_freq = pos_tag_frequency(tagged_tokens)
print('POS Tag Frequency in Sample Review:')
for tag, count in tag_freq.most_common():
    print(f'{tag}: {count}')

POS Tag Frequency in Sample Review:
VBD: 4
.: 3
NN: 2
DT: 2
RB: 2
JJ: 2
NNS: 2
VBG: 1
IN: 1
CD: 1
NNP: 1
CC: 1
WP: 1
PRP: 1


In [8]:
def extract_by_pos(tagged_tokens, pos_prefix):
    """Extract all words matching a given POS tag prefix (e.g. 'NN' for nouns, 'JJ' for adjectives)."""
    return [word for word, tag in tagged_tokens if tag.startswith(pos_prefix)]

nouns = extract_by_pos(tagged_tokens, 'NN')
adjectives = extract_by_pos(tagged_tokens, 'JJ')
verbs = extract_by_pos(tagged_tokens, 'VB')

print('Nouns:', nouns)
print('Adjectives:', adjectives)
print('Verbs:', verbs)

Nouns: ['stay', 'place', 'guests', 'Check-in', 'amenities']
Adjectives: ['cozy', 'smooth']
Verbs: ['Amazing', 'felt', 'was', 'were', 'needed']


In [9]:
def get_pos_tags(text):
    """Return the list of POS tags for a review (words dropped, tags only) plus tagged pairs."""
    if not isinstance(text, str) or text.strip() == '':
        return pd.Series({'tagged_tokens': [], 'pos_tags_only': []})
    tokens = word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    tags_only = [tag for _, tag in tagged]
    return pd.Series({'tagged_tokens': tagged, 'pos_tags_only': tags_only})

# Apply on first 500 rows as a demo (remove .head(500) to run on the full dataset)
sample_df = df.head(500).copy()
pos_results = sample_df['review_text'].apply(get_pos_tags)
df_pos = pd.concat([sample_df[['review_id', 'review_text']], pos_results], axis=1)
df_pos.head(10)

,review_id,review_text,tagged_tokens,pos_tags_only
0,369314882,Amazing stay! The place felt very cozy for 4 g...,"[(Amazing, VBG), (stay, NN), (!, .), (The, DT)...","[VBG, NN, ., DT, NN, VBD, RB, JJ, IN, CD, NNS,..."
1,490116563,It was okay for the price. Location in XIII Au...,"[(It, PRP), (was, VBD), (okay, VBN), (for, IN)...","[PRP, VBD, VBN, IN, DT, NN, ., NN, IN, NNP, NN..."
2,582235668,Loved every minute of it. Our superhost was su...,"[(Loved, VBN), (every, DT), (minute, NN), (of,...","[VBN, DT, NN, IN, PRP, ., PRP$, NN, VBD, JJ, N..."
3,68054683,Decent stay overall. Some things could be impr...,"[(Decent, NNP), (stay, NN), (overall, RB), (.,...","[NNP, NN, RB, ., DT, NNS, MD, VB, VBN, ,, IN, ..."
4,248483824,Reasonable for a short trip. Location in Long ...,"[(Reasonable, JJ), (for, IN), (a, DT), (short,...","[JJ, IN, DT, JJ, NN, ., NN, IN, NNP, NNP, NNP,..."
5,155617131,Decent stay overall. It served its purpose for...,"[(Decent, NNP), (stay, NN), (overall, RB), (.,...","[NNP, NN, RB, ., PRP, VBD, PRP$, NN, IN, PRP$,..."
6,710244614,"Nothing special, but fine. Our superhost was p...","[(Nothing, VBG), (special, JJ), (,, ,), (but, ...","[VBG, JJ, ,, CC, JJ, ., PRP$, NN, VBD, JJ, CC,..."
7,299174484,We had a rough experience. The location in Enc...,"[(We, PRP), (had, VBD), (a, DT), (rough, JJ), ...","[PRP, VBD, DT, JJ, NN, ., DT, NN, IN, NNP, VBD..."
8,22136604,Perfect for our trip. Check-in was smooth and ...,"[(Perfect, NN), (for, IN), (our, PRP$), (trip,...","[NN, IN, PRP$, NN, ., NNP, VBD, JJ, CC, DT, NN..."
9,469473761,Would not recommend. The private room in house...,"[(Would, MD), (not, RB), (recommend, VB), (., ...","[MD, RB, VB, ., DT, JJ, NN, IN, NN, VBD, RB, R..."


In [10]:
all_tags = Counter([tag for row in df_pos['pos_tags_only'] for tag in row])

print('Top 15 POS Tags across dataset:')
for tag, count in all_tags.most_common(15):
    print(f'{tag}: {count}')

Top 15 POS Tags across dataset:
NN: 1963
.: 1575
VBD: 1473
JJ: 1429
IN: 1427
DT: 1113
RB: 822
NNP: 808
PRP: 565
NNS: 464
PRP$: 444
CC: 436
VB: 413
VBN: 393
TO: 285


In [11]:
df_pos.to_csv('POS_Tagged_Airbnb_Reviews.csv', index=False)
print('Saved to POS_Tagged_Airbnb_Reviews.csv')

Saved to POS_Tagged_Airbnb_Reviews.csv
